# Geison no Google Colab

Este é o fluxo oficial e fino para executar o Geison no Colab. O notebook cuida apenas de ambiente, configuração e chamadas ao CLI; toda a lógica científica continua dentro do pacote `geison-qpcr`.

Execute as células em ordem. Em uma sessão já preparada, a primeira célula atualiza o checkout com `git pull`; em uma sessão nova, ela clona a branch `main`.


In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Geison/.git ]; then
  git -C Geison checkout main
  git -C Geison pull --ff-only origin main
else
  git clone --branch main https://github.com/BrunoDCamargo/Geison.git
fi
python -m pip install -e /content/Geison


## Dependências externas

CD-HIT, MAFFT e Primer3 são instalados pelo gerenciador do sistema. O comando `doctor` abaixo confirma o ambiente efetivo antes da análise.


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cd-hit mafft primer3


In [ ]:
%cd /content/Geison
!qpcr-pipeline doctor


## Identificação para o NCBI

Aquisição NCBI ao vivo exige `NCBI_EMAIL`. A API key é opcional e, quando usada, é lida de forma oculta e colocada somente em `NCBI_API_KEY` no ambiente da sessão. Esses valores não entram no YAML nem nos artefatos do Geison.


In [ ]:
import os
from getpass import getpass

ncbi_email = input("NCBI e-mail: " ).strip()
if not ncbi_email:
    raise ValueError("NCBI_EMAIL is required for live NCBI acquisition")
os.environ["NCBI_EMAIL"] = ncbi_email

ncbi_api_key = getpass("NCBI API key (optional; press Enter to skip): " ).strip()
if ncbi_api_key:
    os.environ["NCBI_API_KEY"] = ncbi_api_key
else:
    os.environ.pop("NCBI_API_KEY", None)


## Configuração YAML

O exemplo usa um accession NCBI conhecido e habilita alinhamento e conservação para produzir `report.html`. Troque o target e a seção `input` pelo seu material real antes de uma análise de projeto. O YAML é a fonte de configuração; o notebook não reimplementa nenhum algoritmo científico.


In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/geison_run
cat > /content/geison_run/config.yaml <<'YAML'
target:
  name: SARS-CoV-2-example
input:
  ncbi:
    accessions:
      - NC_045512.2
alignment:
  enabled: true
  threads: 2
conservation:
  enabled: true
  window_size: 100
  step_size: 25
YAML
cat /content/geison_run/config.yaml


## Validar e executar

O `--dry-run` valida configuração, ambiente e plano sem iniciar a análise. A execução real usa um diretório de saída fixo para que os checkpoints possam ser retomados depois.


In [ ]:
!qpcr-pipeline run /content/geison_run/config.yaml --dry-run --outdir /content/geison_run/output


In [ ]:
!qpcr-pipeline run /content/geison_run/config.yaml --outdir /content/geison_run/output


## Retomar uma run

Se a sessão cair ou a execução for interrompida, rode novamente a preparação do ambiente e depois use `--resume` apontando para o mesmo `outdir`. Checkpoints válidos são reutilizados.


In [ ]:
!qpcr-pipeline run /content/geison_run/config.yaml --outdir /content/geison_run/output --resume


## Abrir o relatório

O relatório é uma página HTML interativa com JavaScript. No Colab ele é servido por um pequeno servidor local e aberto em um iframe autenticado, preservando gráfico, valores e interações. O arquivo continua em `/content/geison_run/output/report.html` para download ou cópia para o Google Drive.

Para atualizar o Geison em uma sessão existente, execute novamente a primeira célula; ela usa `git pull --ff-only origin main` e reinstala o pacote em modo editável.


In [ ]:
from pathlib import Path
import socket
import subprocess
import sys
import time
from google.colab import output

report_dir = Path("/content/geison_run/output")
report_path = report_dir / "report.html"
if not report_path.is_file():
    raise FileNotFoundError(f"Report not found: {report_path}")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    probe.bind(("127.0.0.1", 0))
    report_port = probe.getsockname()[1]

report_server = subprocess.Popen(
    [sys.executable, "-m", "http.server", str(report_port), "--bind", "127.0.0.1", "--directory", str(report_dir)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

deadline = time.time() + 5
while time.time() < deadline:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        if probe.connect_ex(("127.0.0.1", report_port)) == 0:
            break
    time.sleep(0.1)
else:
    report_server.terminate()
    raise RuntimeError("Could not start local server for report.html")

output.serve_kernel_port_as_iframe(report_port, path="/report.html", height=900)
